In [5]:
#!pip install cftime netCDF4 xarray pandas numpy

In [1]:
"""
Read all 18 EURO-CORDEX NetCDF4 files from Copernicus C3S.

Requirements:
    pip install netCDF4 xarray cftime numpy pandas

Usage:
    python read_cordex_nc_files.py
    (run from the folder containing the 18 .nc files)
"""

import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

NC_DIR     = "."
OUTPUT_DIR = "output_csv"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# NOTE: Greece uses Eurostat code 'EL', not ISO 'GR'
# UK is present in the files but is not in your EU-26 panel
NUTS_TO_COUNTRY = {
    'AT': 'Austria',     'BE': 'Belgium',      'BG': 'Bulgaria',
    'HR': 'Croatia',     'CZ': 'Czechia',      'DK': 'Denmark',
    'EE': 'Estonia',     'FI': 'Finland',      'FR': 'France',
    'DE': 'Germany',     'EL': 'Greece',       'HU': 'Hungary',
    'IE': 'Ireland',     'IT': 'Italy',        'LV': 'Latvia',
    'LT': 'Lithuania',   'LU': 'Luxembourg',   'NL': 'Netherlands',
    'NO': 'Norway',      'PL': 'Poland',       'PT': 'Portugal',
    'RO': 'Romania',     'SK': 'Slovakia',     'SI': 'Slovenia',
    'ES': 'Spain',       'SE': 'Sweden',
    # Non-EU regions also present in the files
    'AL': 'Albania',     'BA': 'Bosnia',       'CH': 'Switzerland',
    'CY': 'Cyprus',      'IS': 'Iceland',      'LI': 'Liechtenstein',
    'ME': 'Montenegro',  'MK': 'North Macedonia', 'MT': 'Malta',
    'RS': 'Serbia',      'TR': 'Turkey',       'UK': 'United Kingdom',
}

# Your EU-26 countries — using Eurostat NUTS codes (EL for Greece)
EU26 = {'AT','BE','BG','HR','CZ','DK','EE','FI','FR','DE',
        'EL','HU','IE','IT','LV','LT','LU','NL','NO','PL',
        'PT','RO','SK','SI','ES','SE'}


def read_nc(path, freq):
    """
    Opens a NetCDF4 CORDEX file and returns a clean long-format DataFrame.
    Handles the proleptic_gregorian calendar via cftime, and the
    plain Unicode NUTS coordinate (dtype <U2).
    """
    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        ds = xr.open_dataset(path, engine='netcdf4', decode_times=time_coder)
    except AttributeError:
        # Fallback for older xarray versions
        ds = xr.open_dataset(path, engine='netcdf4', use_cftime=True)

    # ── Pick data variable ────────────────────────────────────────────────────
    skip = {'height'}
    candidates = [v for v in ds.data_vars if v not in skip]
    data_var = candidates[0]

    da = ds[data_var]

    # Drop height dim/coord — it causes the off-by-one error in to_dataframe()
    if 'height' in da.dims:
        da = da.isel(height=0)
    da = da.drop_vars('height', errors='ignore')

    # ── Convert to long-format DataFrame ─────────────────────────────────────
    # With height dropped, da has dims (time, nuts) — to_dataframe() works cleanly
    df = da.to_dataframe(name='value').reset_index()

    # ── Decode cftime time objects ────────────────────────────────────────────
    # After use_cftime=True, the 'time' column contains cftime objects
    # Access .year and .month directly — do NOT cast to float
    df['year']  = [t.year  for t in df['time']]
    df['month'] = [t.month for t in df['time']]
    df = df.drop(columns=['time'])

    # ── NUTS codes are already plain strings (dtype <U2) ─────────────────────
    # Just clean any whitespace
    df['nuts0'] = df['nuts'].astype(str).str.strip()
    df = df.drop(columns=['nuts'])

    # ── Add country name ──────────────────────────────────────────────────────
    df['country'] = df['nuts0'].map(NUTS_TO_COUNTRY).fillna(df['nuts0'])

    # Drop unused columns
    df = df.drop(columns=['height'], errors='ignore')

    # Final column order
    if freq == 'yearly':
        df = df.drop(columns=['month'])
        df = df[['nuts0', 'country', 'year', 'value']]
    else:
        df = df[['nuts0', 'country', 'year', 'month', 'value']]

    ds.close()
    return df


# ── FILE CATALOGUE ────────────────────────────────────────────────────────────
FILE_CATALOGUE = [
    ('03_heating_degree_days-projections-yearly',              'hdd_annual',  'yearly'),
    ('04_cooling_degree_days-projections-yearly',              'cdd_annual',  'yearly'),
    ('01_mean_temperature-projections-yearly',                 'tas_annual',  'yearly'),
    ('l1_daily_maximum_temperature-projections-yearly-mean',   'tasmax_mean', 'yearly'),
    ('l1_daily_maximum_temperature-projections-yearly-max',    'tasmax_max',  'yearly'),
    ('l1_daily_maximum_temperature-projections-yearly-min',    'tasmax_min',  'yearly'),
    ('l2_daily_minimum_temperature-projections-yearly-mean',   'tasmin_mean', 'yearly'),
    ('l2_daily_minimum_temperature-projections-yearly-max',    'tasmin_max',  'yearly'),
    ('l2_daily_minimum_temperature-projections-yearly-min',    'tasmin_min',  'yearly'),
    ('03_heating_degree_days-projections-monthly',             'hdd',         'monthly'),
    ('04_cooling_degree_days-projections-monthly',             'cdd',         'monthly'),
    ('01_mean_temperature-projections-monthly',                'tas',         'monthly'),
    ('l1_daily_maximum_temperature-projections-monthly-mean',  'tasmax_mean', 'monthly'),
    ('l1_daily_maximum_temperature-projections-monthly-max',   'tasmax_max',  'monthly'),
    ('l1_daily_maximum_temperature-projections-monthly-min',   'tasmax_min',  'monthly'),
    ('l2_daily_minimum_temperature-projections-monthly-mean',  'tasmin_mean', 'monthly'),
    ('l2_daily_minimum_temperature-projections-monthly-max',   'tasmin_max',  'monthly'),
    ('l2_daily_minimum_temperature-projections-monthly-min',   'tasmin_min',  'monthly'),
]


# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
nc_files = sorted(glob.glob(os.path.join(NC_DIR, "*.nc")))
print(f"Found {len(nc_files)} .nc files\n")

yearly_dfs  = {}
monthly_dfs = {}

for nc_path in nc_files:
    fname = os.path.basename(nc_path)
    print(f"{'='*60}")
    print(f"Reading: {fname}")

    meta = None
    for prefix, col_name, freq in FILE_CATALOGUE:
        if fname.startswith(prefix):
            meta = (col_name, freq)
            break

    if meta is None:
        print(f"  WARNING: no catalogue entry — skipping")
        continue

    col_name, freq = meta

    try:
        df = read_nc(nc_path, freq)
        df = df.rename(columns={'value': col_name})

        print(f"  ✓ {col_name}: {len(df):,} rows | "
              f"years {df.year.min()}–{df.year.max()} | "
              f"nuts codes: {sorted(df.nuts0.unique())}")

        if freq == 'yearly':
            yearly_dfs[col_name] = df
        else:
            monthly_dfs[col_name] = df

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()


# ── MERGE & SAVE: ANNUAL ──────────────────────────────────────────────────────
if yearly_dfs:
    print(f"\n{'='*60}")
    print("Merging annual variables...")

    keys   = list(yearly_dfs.keys())
    merged = yearly_dfs[keys[0]][['nuts0','country','year', keys[0]]].copy()
    for key in keys[1:]:
        right  = yearly_dfs[key][['nuts0','year', key]]
        merged = merged.merge(right, on=['nuts0','year'], how='outer')

    merged = merged.sort_values(['country','year']).reset_index(drop=True)

    # All regions
    p_all = os.path.join(OUTPUT_DIR, 'cordex_rcp45_annual_all.csv')
    merged.to_csv(p_all, index=False)
    print(f"  Saved (all): {p_all}  {merged.shape}")

    # EU-26 only
    eu26_a = merged[merged['nuts0'].isin(EU26)].copy()
    p_eu26 = os.path.join(OUTPUT_DIR, 'cordex_rcp45_annual_eu26.csv')
    eu26_a.to_csv(p_eu26, index=False)
    print(f"  Saved (EU26): {p_eu26}  {eu26_a.shape}")

    print(f"\n  Columns : {list(merged.columns)}")
    print(f"  Years   : {merged.year.min()}–{merged.year.max()}")
    print(f"\n  Sample (Germany, first 6 rows):")
    print(eu26_a[eu26_a['nuts0']=='DE'].head(6).to_string(index=False))


# ── MERGE & SAVE: MONTHLY ─────────────────────────────────────────────────────
if monthly_dfs:
    print(f"\n{'='*60}")
    print("Merging monthly variables...")

    keys     = list(monthly_dfs.keys())
    merged_m = monthly_dfs[keys[0]][['nuts0','country','year','month', keys[0]]].copy()
    for key in keys[1:]:
        right    = monthly_dfs[key][['nuts0','year','month', key]]
        merged_m = merged_m.merge(right, on=['nuts0','year','month'], how='outer')

    merged_m = merged_m.sort_values(['country','year','month']).reset_index(drop=True)
    eu26_m   = merged_m[merged_m['nuts0'].isin(EU26)].copy()
    p_m      = os.path.join(OUTPUT_DIR, 'cordex_rcp45_monthly_eu26.csv')
    eu26_m.to_csv(p_m, index=False)
    print(f"  Saved: {p_m}  {eu26_m.shape}")


# ── THESIS-READY: HDD/CDD ANNUAL ─────────────────────────────────────────────
if 'hdd_annual' in yearly_dfs and 'cdd_annual' in yearly_dfs:
    print(f"\n{'='*60}")
    print("Building thesis-ready HDD/CDD annual panel (EU-26)...")

    panel = yearly_dfs['hdd_annual'][['nuts0','country','year','hdd_annual']].copy()
    panel = panel.merge(
        yearly_dfs['cdd_annual'][['nuts0','year','cdd_annual']],
        on=['nuts0','year'], how='outer'
    )
    if 'tas_annual' in yearly_dfs:
        panel = panel.merge(
            yearly_dfs['tas_annual'][['nuts0','year','tas_annual']],
            on=['nuts0','year'], how='left'
        )

    panel = panel[panel['nuts0'].isin(EU26)].copy()
    panel = panel.sort_values(['country','year']).reset_index(drop=True)

    tp = os.path.join(OUTPUT_DIR, 'hdd_cdd_rcp45_eu26_thesis.csv')
    panel.to_csv(tp, index=False)

    print(f"  Saved  : {tp}")
    print(f"  Shape  : {panel.shape}")
    print(f"  Years  : {panel.year.min()}–{panel.year.max()}")
    print(f"  Countries ({panel.country.nunique()}): {sorted(panel.country.unique())}")
    print(f"\n  Preview (first 10 rows):")
    print(panel.head(10).to_string(index=False))

    print(f"""
NOTE (Chapter 3 data section):
  - Greece uses Eurostat NUTS code 'EL' (not ISO 'GR') — already mapped correctly
  - UK is present in the raw files but excluded from the EU-26 panel
  - HDD/CDD computed using Copernicus/Spinoni methodology
    (base temps differ slightly from Eurostat nrg_chdd_a — document in Ch. 3)
  - To merge with panel_data.csv: join on ['country', 'year']
""")

print(f"{'='*60}")
print("Done. Outputs saved to ./output_csv/")


Found 18 .nc files

Reading: 01_mean_temperature-projections-monthly-rcp_4_5-cclm4_8_17-mpi_esm_lr-r1i1p1-layer-nuts_0-latitude-v1.0.nc
  ✓ tas: 67,044 rows | years 1950–2100 | nuts codes: ['AL', 'AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LI', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UK']
Reading: 01_mean_temperature-projections-yearly-rcp_4_5-cclm4_8_17-mpi_esm_lr-r1i1p1-layer-nuts_0-latitude-v1.0.nc
  ✓ tas_annual: 5,587 rows | years 1950–2100 | nuts codes: ['AL', 'AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LI', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UK']
Reading: 03_heating_degree_days-projections-monthly-rcp_4_5-cclm4_8_17-mpi_esm_lr-r1i1p1-layer-nuts_0-latitude-v1.0.nc
  ✓ hdd: 67,044 rows | years 1950–2100 | nuts codes: ['AL', 'AT', 'BE', 